# v2 crop independent CNN: Colab pipeline

Follows `V2_WORKFLOW.md` (Phases 1 to 8) and `crop_indi_cnn/HANDOFF_v2.md`, **crop track only**.

- Task: independent **crop** classifier, 55 classes, 3 backbones (ResNet-50, EfficientNet-B0, ConvNeXt-Tiny), dataset `DPD_Small_clean_v2`.
- The main run uses the dataset's `train_independent.py` **unchanged**. No data preparation, no own improvements here (those go in a separate folder afterwards).
- Every phase ends with a ✅ "you should see" check. If you don't see it, stop (see the troubleshooting table at the end) and ask in the WhatsApp group. Don't work around it silently.

| Phase | What | Where in this notebook |
|---|---|---|
| 1 | Kickoff: read docs, post STARTING | sections 1 |
| 2 | Environment sanity check | section 2 |
| 3 | Smoke test (about 2 min) | section 3 |
| 4 | Main run (about 1.5 h on a T4) | section 4 |
| 5 | Collect and check outputs | section 5 |
| 6 | Run README inputs (SHA-256, analysis) | section 6 |
| 7 | Commit and push | section 7 |
| 8 | DONE message and checkpoints | section 8 |

**Colab notes:** Runtime > Change runtime type > **T4 GPU**. Keep the tab open during the main run. Results are written straight to Google Drive, so finished backbones survive a disconnect.

## Phase 1: kickoff

Do these outside the notebook first:
1. `git pull` the repo at the commit from the `[v2 dataset] READY` message.
2. Read `V2_WORKFLOW.md`, then `crop_indi_cnn/README.md` and `HANDOFF_v2.md`.
3. Read the dataset's `README.md` ("Quick start", "Shared protocol") and `V2_CHANGES.md` (the cells below print them).
4. **Sync point:** post the STARTING message in the group (the cell below fills it in).

✅ All three owners replied with the **same Kaggle version and commit**. If anyone's differs, fix it now, not after training.

**Rules that keep the tracks comparable:** dataset loader and augmentation unchanged; never use `test_seen` / `test_unseen` to choose epochs, learning rates or models; best epoch = highest harmonic mean of `val` and `val_unseen` Top-1 (patience 10); seed 42, batch 64, up to 30 epochs, AdamW wd 0.05; pinned ImageNet-1k weights; main run first and unchanged; never edit or delete the v1 results folder `output/crop_independent/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ----------------------------- EDIT THESE -----------------------------
DRIVE_DATASET  = "/content/drive/MyDrive/<path>/DPD_Small_clean_v2"   # v2 dataset folder in your Drive
KAGGLE_VERSION = "<n>"      # dataset version number the team agreed on
REPO_COMMIT    = "<hash>"   # commit from the [v2 dataset] READY message
# ----------------------------------------------------------------------

import os
assert "<" not in DRIVE_DATASET, "Edit DRIVE_DATASET above."
assert os.path.isdir(DRIVE_DATASET), f"Dataset folder not found: {DRIVE_DATASET}"
assert "<" not in KAGGLE_VERSION and "<" not in REPO_COMMIT, "Fill in KAGGLE_VERSION and REPO_COMMIT (needed for the group messages and the README)."

DATA = "/content/DPD_Small_clean_v2"                               # local copy (Drive is slow to read)
OUT  = "/content/drive/MyDrive/DPD_runs/v2/crop_independent_v2"    # results go to Drive
print("DRIVE_DATASET:", DRIVE_DATASET)
print("DATA:", DATA)
print("OUT :", OUT)

In [ ]:
!cp -r "{DRIVE_DATASET}" /content/

assert os.path.exists(f"{DATA}/label_maps.json"), "label_maps.json missing: wrong folder or incomplete copy."
assert os.path.exists(f"{DATA}/train_independent.py"), "train_independent.py missing from the dataset folder."
print(sorted(os.listdir(DATA)))

In [ ]:
# Post this in the WhatsApp group (sync point)
print(f"[v2 crop_independent] STARTING – Kaggle v{KAGGLE_VERSION}, commit {REPO_COMMIT}")

In [ ]:
# Read: dataset README "Quick start" and "Shared protocol"
def show_section(path, keywords, max_lines=80):
    if not os.path.exists(path):
        print(f"[missing] {path}"); return
    lines = open(path, encoding="utf-8").read().splitlines()
    for kw in keywords:
        start = next((i for i, l in enumerate(lines) if l.lstrip().startswith("#") and kw.lower() in l.lower()), None)
        if start is None:
            print(f"[no section matching '{kw}' in {os.path.basename(path)}]"); continue
        level = len(lines[start]) - len(lines[start].lstrip("#"))
        end = len(lines)
        for j in range(start + 1, len(lines)):
            if lines[j].startswith("#") and (len(lines[j]) - len(lines[j].lstrip("#"))) <= level:
                end = j; break
        print("=" * 80)
        print("\n".join(lines[start:min(end, start + max_lines)]))
        print()

show_section(f"{DATA}/README.md", ["Quick start", "Shared protocol"])

In [ ]:
# Read: V2_CHANGES.md (section 7 describes what each data-preparation script did; useful for the report)
!cat {DATA}/V2_CHANGES.md

## Phase 2: environment sanity check

Colab replaces the Kaggle "Add Input / Settings" steps: pick the T4 runtime (internet is on by default, needed for the pretrained weights).

✅ You should see a GPU name (e.g. `Tesla T4`), `cuda True`, and exactly:
```
{'train': 6995, 'val': 773, 'val_unseen': 227, 'test_seen': 2000, 'test_unseen': 1448}
```

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install timm

In [ ]:
import sys
sys.path.insert(0, DATA)
import torch, timm, dataset

print("DATA:", DATA)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())
sizes = {s: len(dataset.read_split(s)) for s in dataset.SPLITS}
print(sizes)

EXPECTED = {"train": 6995, "val": 773, "val_unseen": 227, "test_seen": 2000, "test_unseen": 1448}
assert torch.cuda.is_available(), "cuda False: Runtime > Change runtime type > T4 GPU"
assert sizes == EXPECTED, "Split sizes differ: wrong dataset version (v1 attached?). Stop and post in the group."
print("OK: GPU available and split sizes match v2.")

## Phase 3: smoke test (about 2 minutes)

One epoch, 3 batches, ResNet-50 only. The accuracy numbers are meaningless; only check that the pipeline runs.

✅ You should see a line `split sizes: {'train': 6995, ...} | device: cuda`, one epoch line like `[ResNet50] ep 01 | train loss … | val … | val_unseen … | H …`, then a one-row summary table, and `/content/smoke/` containing `results.json`, `summary.csv`, confusion matrices and prediction files.

In [ ]:
!python {DATA}/train_independent.py --task crop --out /content/smoke --models ResNet50 --epochs 1 --limit-batches 3 --workers 2

In [ ]:
smoke = "/content/smoke"
names = sorted(os.listdir(smoke))
print(names)
assert "results.json" in names and "summary.csv" in names, "smoke run did not write results.json / summary.csv"
assert any("confusion_matrix" in n for n in names), "no confusion matrices in smoke output"
assert any("predictions" in n for n in names), "no prediction files in smoke output"
print("Smoke test OK.")

In [ ]:
# Delete the smoke folder, as the workflow says
!rm -rf /content/smoke

## Phase 4: main run (about 1.5 h on a T4)

The command is the handoff's Colab command, script unchanged, `--workers 2`, output on Drive.

What is normal while it runs:
- One line per epoch with `val`, `val_unseen` and `H`. **`val_unseen` is small (227 images) and jumps a few points between epochs.** That is expected; the best epoch is chosen on `H`.
- Early stopping prints `early stop at epoch N` when `H` hasn't improved for 10 epochs. Runs often end before epoch 30.
- `results.json` is rewritten after each backbone, so finished backbones are safe even if a later one crashes.

✅ The run finishes and prints the final summary table with **3 rows**.

Keep this notebook's outputs: it must be saved **with outputs** as `crop_independent_cnns_v2.ipynb`.

In [ ]:
!python {DATA}/train_independent.py --task crop --workers 2 --out {OUT}

**If the run stopped half-way** (Colab disconnect, quota): finished backbones are in `results.json`. Reconnect, re-run the cells in sections 1 and 2, then rerun only the missing backbone(s), one `--models` name at a time, into the **same** `--out` folder. Uncomment and adapt the line below, and write it in your README's process log. Post in the group that you had to resume.

In [ ]:
# !python {DATA}/train_independent.py --task crop --workers 2 --out {OUT} --models ConvNeXt-Tiny

## Phase 5: collect and check the outputs

The results are already in Drive at `OUT`. This section checks them against the workflow and the handoff (section 4).

✅ The file count is 47; every model has a best epoch; seen > unseen; nothing is exactly 0 or 100; `train_clean_top1` is below 100%.

In [ ]:
import json
import pandas as pd

MODELS = ["ResNet50", "EfficientNet-B0", "ConvNeXt-Tiny"]
SPLITS = ["val", "val_unseen", "seen", "unseen"]

# 1) File count: 47 = 3 x 14 (4 confusion CSVs + 4 PNGs + 4 prediction CSVs + history + curves) + 5 shared files
expected = {"config.json", "results.json", "summary.csv", "all_models_accuracy.png", "harmonic_mean.png"}
for m in MODELS:
    expected |= {f"{m}_history.csv", f"{m}_curves.png"}
    for s in SPLITS:
        expected |= {f"{m}_{s}_confusion_matrix.csv", f"{m}_{s}_confusion_matrix.png", f"{m}_predictions_{s}.csv"}

actual = {f for f in os.listdir(OUT) if not f.endswith(".pt") and f != "README.md"}
print("files (excluding .pt and README.md):", len(actual), "| expected:", len(expected))
print("missing:", sorted(expected - actual) or "none")
print("unexpected extra:", sorted(actual - expected) or "none")
assert len(expected) == 47
assert not (expected - actual), "Missing output files. Stop; see troubleshooting."

# 2) Checkpoints exist
for m in MODELS:
    assert os.path.exists(f"{OUT}/{m}_best.pt"), f"{m}_best.pt missing"

# 3) config.json: dataset, pinned timm_ids, GPU name
cfg = json.load(open(f"{OUT}/config.json"))
print(json.dumps(cfg, indent=2))
assert cfg.get("dataset") == "DPD_Small_clean_v2", "config.json dataset is not DPD_Small_clean_v2"

# 4) summary.csv has 3 rows
summary = pd.read_csv(f"{OUT}/summary.csv")
display(summary)
assert len(summary) == 3, "summary.csv should have 3 rows"

In [ ]:
# 5) Sanity check of the numbers (from V2_WORKFLOW.md) plus clean-train check (HANDOFF section 4)
res = json.load(open(f"{OUT}/results.json"))
problems = []
for m, x in res.items():
    s = x.get("pair", x)
    seen, unseen, H = s["seen"]["1"], s["unseen"]["1"], s["harmonic"]["1"]
    print(f"{m:16s} best epoch {x['best_epoch']:>2} (ran {x['epochs_run']:>2}) | seen {seen:.1f} | unseen {unseen:.1f} | H {H:.1f} "
          f"| train_clean_top1 {x['train_clean_top1']:.1f} | timm_id {x['timm_id']}")
    if not seen > unseen: problems.append(f"{m}: seen <= unseen")
    if any(v in (0, 0.0, 100, 100.0) for v in (seen, unseen, H)): problems.append(f"{m}: a headline number is exactly 0 or 100")
    if not x["train_clean_top1"] < 100: problems.append(f"{m}: train_clean_top1 is 100% (is augmentation on?)")
print("\nProblems:", problems or "none")
if problems:
    print("Stop, don't push: post the summary table in the group.")

In [ ]:
# Results table (Top-1, %), copy into the README. For the v1 comparison use seen_excl_heldout, not seen.
rows = []
for m, r in res.items():
    rows.append({
        "Model": m, "best_epoch": r["best_epoch"], "epochs_run": r["epochs_run"],
        "val": r["val"]["1"], "val_unseen": r["val_unseen"]["1"],
        "seen": r["seen"]["1"], "seen_excl_heldout": r["seen_excl_heldout"]["1"],
        "unseen": r["unseen"]["1"], "H top1": r["harmonic"]["1"], "H top5": r["harmonic"]["5"],
        "seen per-class": r["seen"]["per_class_top1"], "unseen per-class": r["unseen"]["per_class_top1"],
    })
display(pd.DataFrame(rows).round(2))

In [ ]:
# Handoff check: train_acc_aug should rise more slowly than in v1 (augmentation on). Look at the curves.
from IPython.display import Image, display as ipy_display
for m in MODELS:
    print(m)
    ipy_display(Image(f"{OUT}/{m}_curves.png", width=800))

for m in MODELS:
    h = pd.read_csv(f"{OUT}/{m}_history.csv")
    print(m, "history columns:", list(h.columns))

## Phase 6: inputs for your run README

Copy `crop_indi_cnn/README_TEMPLATE.md` into the results folder as `README.md` and fill in **every** section: environment, exact commands, pipeline (plus any change from the protocol, normally none), process log (smoke test, crashes, resumes), training behaviour, results (from `summary.csv`, plus the v1 comparison using `seen_excl_heldout`), error analysis, checkpoints.

The cells below print the values you need.

In [ ]:
# Environment for the README
import platform, subprocess
print("Python:", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Kaggle dataset version:", KAGGLE_VERSION, "| repo commit:", REPO_COMMIT)
print("Workers: 2 | Command: python train_independent.py --task crop --workers 2 --out", OUT)

In [ ]:
# Training behaviour: best epoch, time per epoch, train vs val gap
for m in MODELS:
    h = pd.read_csv(f"{OUT}/{m}_history.csv")
    sec = [c for c in h.columns if "sec" in c.lower()]
    print(f"\n{m}: {len(h)} epochs | best epoch (results.json) {res[m]['best_epoch']}")
    if sec: print(f"  mean seconds/epoch: {h[sec[0]].mean():.0f}")
    display(h.tail(3).round(3))

In [ ]:
# Error analysis for the report (HANDOFF section 6): per-class unseen accuracy; tomato is 33% of test_unseen
for m in MODELS:
    print("=" * 70); print(m)
    r = res[m]
    print(f"seen: overall {r['seen']['1']:.1f} vs per-class {r['seen']['per_class_top1']:.1f}   |   "
          f"unseen: overall {r['unseen']['1']:.1f} vs per-class {r['unseen']['per_class_top1']:.1f}")
    p = pd.read_csv(f"{OUT}/{m}_predictions_unseen.csv")
    if not {"true", "pred1"} <= set(p.columns):
        print("columns:", list(p.columns)); continue
    p["correct"] = p["true"] == p["pred1"]
    g = p.groupby("true")["correct"].agg(acc="mean", n="size")
    g["acc"] *= 100
    g["share_%"] = g["n"] / g["n"].sum() * 100
    print("largest share of test_unseen:")
    display(g.sort_values("n", ascending=False).head(3).round(1))
    print("worst 8 classes (n >= 10):")
    display(g[g["n"] >= 10].sort_values("acc").head(8).round(1))
    key = g[g.index.astype(str).str.contains("tomato|cherry|cucumber|pepper", case=False)]
    print("tomato / cherry / cucumber / pepper:")
    display(key.round(1))

In [ ]:
# Checkpoint list for the README: file, size, SHA-256, epoch, val H. Fill the "date sent" column in Phase 8.
!ls -lh {OUT}/*.pt
!sha256sum {OUT}/*.pt

for m in MODELS:
    ck = torch.load(f"{OUT}/{m}_best.pt", map_location="cpu")
    print(f"{m:16s} epoch {ck['epoch']:>2} | val_H {ck['val_H']:.2f} | val_top1 {ck['val_top1']:.2f} | val_unseen_top1 {ck['val_unseen_top1']:.2f} | {ck['dataset']} | {ck['timm_id']}")
    mdl = timm.create_model(ck["timm_id"], pretrained=False, num_classes=ck["num_classes"])
    mdl.load_state_dict(ck["state_dict"])
print("All 3 checkpoints reload OK.")

In [ ]:
# ✅ No <placeholder> text left in the README (heuristic: angle-bracket words)
import re
readme = f"{OUT}/README.md"
if not os.path.exists(readme):
    print("README.md not in the results folder yet: copy README_TEMPLATE.md there as README.md and fill it in.")
else:
    hits = re.findall(r"<[A-Za-z][^<>\n]{0,40}>", open(readme, encoding="utf-8").read())
    print("possible placeholders left:", hits or "none")

## Phase 7: commit and push

Do these on your own computer (not in Colab).

1. Download the results from Drive (without the `.pt` files) with the cell below, and put them in the repo at `crop_indi_cnn/output/crop_independent_v2/`.
2. Save this notebook **with outputs**: File > Download > Download .ipynb, then save it as `crop_indi_cnn/crop_independent_cnns_v2.ipynb`.
3. Do **not** touch the v1 folder `output/crop_independent/`.

```bash
git pull
git switch -c v2/crop
git add crop_indi_cnn/output/crop_independent_v2/ crop_indi_cnn/crop_independent_cnns_v2.ipynb
git status                                  # check: no .pt files listed (they are git-ignored)
git commit -m "Add v2 crop results (Kaggle dataset v<n>)"
git push -u origin v2/crop
```
Then open a pull request into `master` (or merge, whichever the team agreed).

✅ `git status` is clean, the branch is on GitHub, and no `.pt` file was committed.

In [ ]:
# Zip the results WITHOUT the checkpoints and download them
!cd /content/drive/MyDrive/DPD_runs/v2 && zip -r /content/crop_independent_v2.zip crop_independent_v2 -x "*.pt"
from google.colab import files
files.download("/content/crop_independent_v2.zip")

## Phase 8: tell the group, then send the checkpoints

1. Post the DONE message (the cell below fills in the numbers). Replace `<hash>` with the commit hash of your push. "Best" is the backbone with the highest Top-1 harmonic mean.
2. **Right after it, send the three `.pt` files** to the group: attach > **Document** (not Gallery / Photo / Video), so the file arrives unchanged. Get them from the Drive folder `DPD_runs/v2/crop_independent_v2/`.
3. Fill the "date sent" column in your README's checkpoint table and push that small change.

✅ The group has your message, followed by every checkpoint file.

In [ ]:
best = max(res, key=lambda k: res[k]["harmonic"]["1"])
b = res[best]
print(f"""[v2 crop_independent] DONE
What: independent crop CNN, 3 backbones, shared v2 protocol
Best (Top-1): {best} – seen {b['seen']['1']:.1f} / unseen {b['unseen']['1']:.1f} / H {b['harmonic']['1']:.1f}
Pushed: crop_indi_cnn/output/crop_independent_v2/ (commit <hash>)
Checkpoints: ResNet50_best.pt, EfficientNet-B0_best.pt, ConvNeXt-Tiny_best.pt (sent below; SHA-256 in README)
Notes: <anything off-protocol, problems, or "none">""")

## If something goes wrong

| Symptom | Likely cause | Fix |
|---|---|---|
| `AssertionError` on the dataset folder / `label_maps.json` | wrong `DRIVE_DATASET` path or incomplete copy | fix the path, re-run the copy cell |
| Split sizes differ from 6995 / 773 / 227 / 2000 / 1448 | wrong dataset version (v1 copied) | use the v2 version; post in the group if unsure |
| Error downloading pretrained weights (timm / Hugging Face) | no internet in the runtime | check the Colab connection and rerun |
| `cuda False` | no GPU selected | Runtime > Change runtime type > T4 GPU, then re-run from section 1 |
| `CUDA out of memory` | GPU too small or shared | add `--batch-size 32`; write it in the README as a deviation and say so in the DONE message |
| DataLoader worker errors or freezing | too many workers | `--workers 2` (or `0` to debug) |
| Run stopped half-way | Colab disconnect or quota | finished backbones are in `results.json`; rerun only the missing ones with `--models <name>` into the same `--out` (see the resume cell), then post in the group |
| `.pt` shows up in `git status` | file outside the ignored patterns | don't add it; checkpoints go to WhatsApp only |
| Numbers look wrong (0%, 100%, unseen > seen) | something is off | stop, don't push; post the summary table in the group |